In [ ]:
# Bi-LSTM
import torch
import torch.nn as nn
import torch.nn.functional as F

embed_size = 10
hidden_size = 12
# input_size : 입력되는 단어벡터의 차원
# hidden_size : LSTM 내부적으로 사용하는 은닉상태
# num_layers : LSTM을 몇겹으로 쌓는지
# bidirectrinal : True , 정방향과 역방향 두개의 LSTM이 동시에 작동  
#   최종 벡터가 두 방향벡터의 결과를 이어붙이므로 hidden_size*2

elmo_bilstm = nn.LSTM(input_size=embed_size, hidden_size=hidden_size, num_layers=2, batch_first=True, bidirectional=True)

# 가상의 데이터 입력 (문장 1개, 단어 5개, 각 단어 10차원)
dumy_input = torch.randn(1, 5, embed_size)
output, (hn, cn) = elmo_bilstm(dumy_input)
print('입력텐서 차원:', dumy_input.shape)
print('출력텐서 차원:', output.shape)
# 마지막 출력이 24인 이유는 hidden_size : 12 --> 정방향,역방향 = 24차원

입력텐서 차원: torch.Size([1, 5, 10])
출력텐서 차원: torch.Size([1, 5, 24])


In [6]:
# bank의 벡터값이 문맥에 따라서 다르게 나오는지 코사인 유사도로 확인
# F.cosine_similarity(텐서1, 텐서2, dim) : dim에 해당하는 차원을 기준으로 계산
# 임시 단어사전 구성 10차원 정적 임베딩
word_to_idx = {'I':0, 'deposited':1, 'money':2, 'in':3, 'the':4, 'bank':5, 'walked':6, 'by':7, 'river':8, '.':9}
static_embedding = nn.Embedding(num_embeddings=10, embedding_dim=embed_size)

# 문장 A: I deposited money in the bank .
sentence_A_idx = torch.tensor([[word_to_idx[w] for w in ['I', 'deposited', 'money', 'in', 'the', 'bank', '.']]])
# 문장 B: I walked by the river bank .
sentence_B_idx = torch.tensor([[word_to_idx[w] for w in ['I', 'walked', 'by', 'the', 'river', 'bank', '.']]])

# 1. 정적 임베딩 통과 (문맥반영 안됨)
emb_A = static_embedding(sentence_A_idx)
emb_B = static_embedding(sentence_B_idx)

# 2. bi-lstm (양방향 적용된 임베딩)
elmo_out_A,_ = elmo_bilstm(emb_A)
elmo_out_B,_ = elmo_bilstm(emb_B)

# 3. 두 문장에서 bank 단어의 위치(인덱스) 둘다 5
# 정적 임베딩에서 bank 벡터간 유사도 (A 문장의 bank vs B 문장의 bank)
static_bank_A = emb_A[0,5,:]
static_bank_B = emb_B[0,5,:]
static_sim = F.cosine_similarity(static_bank_A.unsqueeze(0), static_bank_B.unsqueeze(0))

# elmo 임베딩에서 bank
elmo_bank_A = elmo_out_A[0,5,:]
elmo_bank_B = elmo_out_B[0,5,:]
elmo_sim = F.cosine_similarity(elmo_bank_A.unsqueeze(0), elmo_bank_B.unsqueeze(0))

print('정적 임베딩:', static_sim.item())
print('동적 임베딩 :', elmo_sim.item())

정적 임베딩: 1.0
동적 임베딩 : 0.9863191246986389


In [8]:
# feature-based 전이학습
# elmo가 생성된 문맥을 반영한 벡터를 기존 딥러닝 분류기에 어떻게 전이학습 가져다 쓰는지 concat연산을 통해 확인
# 기존 10차원이고 elmo 벡터가 24차원이면 둘을 붙이면 총 34차원의 풍부한 특징 벡터 1개 탄생

original_feature = static_bank_A
elmo_feature = elmo_bank_A

# feature-base 전이학습 : 기존 벡터에 elmo 벡터를 이어붙임
combined_feature = torch.cat([original_feature,elmo_feature])

original_feature.shape, elmo_feature.shape, combined_feature.shape

(torch.Size([10]), torch.Size([24]), torch.Size([34]))